# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [1]:
!docker info

Emulate Docker CLI using podman. Create /etc/containers/nodocker to quiet msg.
host:
  arch: amd64
  buildahVersion: 1.45.1
  cdiSpecDirs:
  - /etc/cdi
  - /var/run/cdi
  cgroupControllers:
  - cpu
  - memory
  - pids
  cgroupManager: systemd
  cgroupVersion: v2
  conmon:
    package: conmon-1:2.2.1-1
    path: /usr/bin/conmon
    version: 'conmon version 2.2.1, commit: c8cc2c4db27531bd4e084ce7857f73cd21ee639d'
  cpuUtilization:
    idlePercent: 73.62
    systemPercent: 7.67
    userPercent: 18.71
  cpus: 4
  databaseBackend: sqlite
  distribution:
    distribution: endeavouros
    version: unknown
  eventLogger: journald
  freeLocks: 2044
  hostname: SAM-EOS
  idMappings:
    gidmap:
    - container_id: 0
      host_id: 1000
      size: 1
    - container_id: 1
      host_id: 100000
      size: 65536
    uidmap:
    - container_id: 0
      host_id: 1000
      size: 1
    - container_id: 1
      host_id: 100000
      size: 65536
  kernel: 7.2.7-arch1-1
  linkmode: dynamic
  logDriver: j

Disclaimer: I use podman, but pretty much all commands are the same, so I'll just run your commands like they are (e.g `docker info`), while providing mine as `podman info`. If you want to run them for testing, you should be able to just switch the `podman` part to `docker` without touching the rest of the command.

### What is a container?

A namespace mapping to create a separated space for a process (or a part of the process tree).
This then acts like a virtual machine, but shares the kernel with the underlying operating system.

### Why do we use containers?

Containers basically allow a developer to argue 'It runs on my machine' when shipping software.
Said in other words, it allows us to pack a whole bunch of dependencies into one executable scaffold.

### What is a docker image?

A mapping instruction used to create a namespace mapping to isolate some processes from the namespace of the underlying OS.
It is usually layered, meaning that images are build onto one another.

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly

### Run your first docker container

In [ ]:
!docker run hello-world

### Find the container ID

In [ ]:
!podman ps -a -f "ancestor=hello"

### Delete the container again, give prove its deleted

In [ ]:
!podman ps -a -f "ancestor=hello" --format "{{.ID}}" | xargs podman rm 2>/dev/null

In [ ]:
!podman ps -a

### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. `syay -S fastqc` then yes to all prompts
2. `fastqc --noextract -o fastqcout ../day_02/fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz`
...

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [ ]:
# pull the container
!podman pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

In [ ]:
# run the container and save the results to a new "fastqc_results" directory
!podman images -f reference=fastqc --format "{{.ID}}" | xargs -I {} podman run --rm -v ../day_02/fetchngs_out/fastq/:/fastq  -v ./fastqcout:/fastqcout {} fastqc --noextract -o /fastqcout /fastq/SRX19144486_SRR23195516_1.fastq.gz

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

In this specific scenario running it locally was easier, as there weren't really any dependencies to install.
However to keep my computer clean I usually don't run course software on my host system, so when I get the choice,
I usually run everything containerized.

### What would you say, which approach is more reproducible?

In this scenario it does not really make much difference, but of course with large dependency trees, mutable systems become an issue, so a container usually provides a more stable environment.

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

Not done due to missing files.

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor

### Explain the RUN and ENV lines you added to the file

The `RUN` flag in Dockerfiles allows us to run commands that get baked into the image.  
The `ENV` flag allows to set environment variables. While this could be done using `RUN`, `ENV` conveniently handles exporting and other pitfalls.


In [4]:
# build the docker image
!podman build -t cowsay -f my_dockerimage .

In [ ]:
# make sure that the image has been built
!podman images -f reference=cowsay --quiet | wc -l | xargs test 1 -eq

In [ ]:
# run the docker file 
!podman run --rm cowsay

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image
!cat ./salmon_docker

In [ ]:
# build the image
!podman build --no-cache -t salmon -f ./salmon_docker .

In [1]:
# run the docker image to give out the version of salmon
!podman run --rm salmon -v

## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

Find the salmon docker image online and run it on your computer.

In [ ]:
!podman pull docker.io/getwilds/salmon

What is https://biocontainers.pro/ ?

Seems like nf-core, but for container images

## Are there other ways to create Docker (or Apptainer) images?

A lot: building, commiting, composing, loading, importing, ...

What is https://seqera.io/containers/ ?

A service to create custom repository links for us to get containers with just the tools we need.